# HARPS grouping V1

Groups the HARPS spectra by star using the reduced target name only.
Same logic as the old `sanitise_reduce.ipynb` → `grouping.ipynb` → `grouping_V2.ipynb`:

1. Sanitise the archive name (catalogue-specific fixes)
2. Reduce it (lowercase, no spaces)
3. Normalise known aliases and catalogue formatting
4. One group per normalised reduced name

Input: `state.sqlite` from `scripts/harps_fetch.py`. Output: `<data_dir>/grouping/v1/`.

In [ ]:
'''
Updated 29-10-2026

Form groups for every individual target (HARPS)
'''
import json
import re
import sqlite3
from pathlib import Path
from statistics import mode

import pandas as pd


def find_repo(start=Path.cwd()):
    '''Walk up from the notebook until config/harps.json is found.'''
    for folder in [start, *start.parents]:
        if (folder / 'config' / 'harps.json').exists():
            return folder
    raise FileNotFoundError('config/harps.json not found above ' + str(start))


config = json.loads((find_repo() / 'config' / 'harps.json').read_text())
DATA_DIR = Path(config['data_dir'])
DB_PATH = DATA_DIR / 'state.sqlite'
OUT_DIR = DATA_DIR / 'grouping' / 'v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
### Load metadata ###
# Spectra downloaded and covering both Ca II K and H (HARPS equivalent of df_ca_doublet.pkl)
db = sqlite3.connect(f'file:{DB_PATH.as_posix()}?mode=ro', uri=True)  # read-only: safe while the download runs
metadata = pd.read_sql_query(
    "SELECT dp_id, target_name, s_ra, s_dec, t_min, snr, proposal_id, file "
    "FROM spectra WHERE status = 'done' AND covers_K = 1 AND covers_H = 1",
    db,
)
db.close()

print(f'Loaded metadata: {len(metadata)} spectra, {metadata["target_name"].nunique()} raw target names')

In [ ]:
### Sanitise + reduce names ###

def exceptions(name):
    '''Catalogue-specific formatting of a target name.'''

    # Case for BD catalogue
    if name.startswith('BD'):
        return name

    # Case for ex. CD-234-2432
    if name.startswith('CD'):
        return name[:3] + name[3:].replace('-', '')

    # Case for J###-## and G##-##
    if len(name) > 1 and name[0] in ('J', 'G') and name[1].isnumeric():
        return name

    # Case for CPD-###
    if name.startswith('CPD'):
        return name

    # Case for LP catalogue
    if len(name) > 3 and name.startswith('LP') and name[2].isnumeric():
        return name

    # Case for SERAM catalogue
    if len(name) > 5 and name.startswith('SERAM'):
        return name[:7].replace('-', '') + name[7:]

    # Case of SW catalogue
    if name.startswith('SW'):
        return name

    # Case for TYC catalogue
    if name.startswith('TYC'):
        return name[:5].replace('-', '') + name[5:]

    return name.replace('-', ' ')


def sanitise(header):
    '''Archive name -> sanitised name.'''
    # getting rid of 'A_' in example A_HD45637 (every name starting with a letter up to F + '_')
    if len(header) > 1 and header[0].lower() in 'abcdef' and header[1] == '_':
        header = header[2:]
    return exceptions(header).replace('_', ' ')


def reduce(sanitised):
    '''Sanitised name -> reduced name (lowercase, no spaces).'''
    return sanitised.lower().replace(' ', '')


metadata.insert(1, 'Sanitised', metadata['target_name'].fillna('').map(sanitise))
metadata.insert(2, 'Reduced', metadata['Sanitised'].map(reduce))

print(f'Reduced names: {metadata["Reduced"].nunique()} unique')

In [ ]:
### Alias normalization ###
# Maps non-canonical reduced names → canonical form.
# Keys = aliases confirmed in the old (UVES) data; values = preferred name.
_ALIAS_MAP = {
    'betapic':      'hd39060',
    'betapictoris': 'hd39060',
    'spica':        'alphavir',
    'tetaql':       'thetaaql',
    '49cet':        '49ceti',
    'alphalyr':     'vega',
    'hd216956':     'fomalhaut',
    'betelgeuse':   'hd39801',
    'hd143275':     'deltasco',
}
_HD_ZERO  = re.compile(r'^(hd)0+(\d+)(.*)$')
_HIP_ZERO = re.compile(r'^(hip)0+(\d+)$')
_CS_ZERO  = re.compile(r'^(cs\d{5})0(\d{3,4})$')
_CD_D     = re.compile(r'^(cd-\d+)d(\d+)$')


def normalize_reduced(name):
    '''Return the canonical reduced name, resolving known aliases and catalog formatting.'''
    if name in _ALIAS_MAP:
        return _ALIAS_MAP[name]
    m = _HD_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2) + m.group(3)
    m = _HIP_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2)
    if name.startswith('he~'):
        return 'he' + name[3:]
    m = _CS_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2)
    m = _CD_D.match(name)
    if m:
        return m.group(1) + m.group(2)
    return name


before_unique = metadata['Reduced'].nunique()
metadata['Reduced'] = metadata['Reduced'].map(normalize_reduced)
after_unique = metadata['Reduced'].nunique()
print(f'Alias normalization: {before_unique} → {after_unique} unique names ({before_unique - after_unique} merged)')

In [ ]:
### Get groups according to normalised reduced name ###
name_to_id = {name: i for i, name in enumerate(sorted(metadata['Reduced'].unique()))}
metadata['New Groups'] = metadata['Reduced'].map(name_to_id)

# Get all information for all the different groups
grouped = metadata.groupby('New Groups')
groups = grouped.first()
for col in ['target_name', 'Sanitised', 'Reduced']:
    groups[col] = grouped[col].agg(mode)
groups['Nb Obs'] = grouped.size()
groups = groups.drop(columns=['dp_id', 's_ra', 's_dec', 't_min', 'snr', 'proposal_id', 'file'])
groups = groups.reset_index(drop=False)

sizes = groups['Nb Obs']
print(f'Groups: {len(groups)}')
print(f'Groups with < 3 spectra: {(sizes < 3).sum()}')
print(f'Groups with < 2 spectra: {(sizes < 2).sum()}')
print(groups.sort_values('Nb Obs', ascending=False).head(10))

In [ ]:
### Save outputs ###
full_path = OUT_DIR / 'harps_full_metadata_v1.pkl'
groups_path = OUT_DIR / 'harps_groups_v1.pkl'

metadata.to_pickle(full_path)
groups.to_pickle(groups_path)
print(f'Saved: {full_path}')
print(f'Saved: {groups_path}')